# Electrode Map Generator
This interface automatically reads electrode layouts from DXF files and maps channels to PCB/headstage channels using the `AssemblyRouter`.

**Return**: \(3 \times N_{\text{channels}}\) matrix (X-location, Y-location, PCB/Headstage channel)

In [ ]:
import csv
from pathlib import Path
import ezdxf
import ipywidgets as widgets
import numpy as np
from ipyfilechooser import FileChooser
import tempfile
from collections import defaultdict


import utils
from assembly_routing import AssemblyRouter, ComponentDefinition

In [2]:
doc = ezdxf.readfile("../electrode_map_example/example1/50um_pitch_compatible_with_FPC.dxf")
#doc = ezdxf.readfile("../electrode_map_example/example2/Xiaomin_electrode_v2.dxf")
layers = sorted({e.dxf.layer for e in doc.modelspace()})
print(layers)

['L2D0_etching2', 'L3D0_etching3', 'electrode', 'etching', 'metalization', 'polyimide', 'polyimide_etch_borders']


In [3]:
# give the layer name of the electrode contours in the DXF file
# if you don't know, give None and instead the number of contours to extract (the largest ones will be used)
layer_name = "electrodes"
# if you don't know the layer name, give None and instead the number of contours to extract (the one closest to the origin will be used)
# ATTENtiON: we assume that the last contour is a hook loop, which we will remove from the electrode layout
num_contours = 64

In [ ]:
import ipywidgets as widgets
from IPython.display import display

dxf_chooser = widgets.FileUpload(
    accept=".dxf",
    multiple=True,
    description="Select DXF files",
)

electrode_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select electrode JSON files",
)

flex_pcb_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select flex PCB JSON files",
)

rigid_pcb_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select rigid PCB JSON files",
)

headstage_chooser = widgets.FileUpload(
    accept=".json",
    multiple=True,
    description="Select headstage JSON files",
)

assembly_chooser = widgets.FileUpload(
    accept=".csv",
    multiple=True,
    description="Select assembly CSV files",
)

display(dxf_chooser)
display(electrode_chooser)
display(flex_pcb_chooser)
display(rigid_pcb_chooser)
display(headstage_chooser)
display(assembly_chooser)

FileUpload(value=(), accept='.dxf', description='Select DXF files', multiple=True)

FileUpload(value=(), accept='.json', description='Select probe JSON files', multiple=True)

FileUpload(value=(), accept='.json', description='Select headstage JSON files', multiple=True)

FileUpload(value=(), accept='.csv', description='Select assembly CSV files', multiple=True)

In [ ]:


run_button = widgets.Button(description="Generate Channel Map", button_style="success")
output = widgets.Output()

state = {"channel_map": None, "dxf_file": None}


def extract_uploaded_files(upload_widget, temp_dir):
    """Saves uploaded widget files to a temporary directory and returns their Paths."""
    saved_paths = []
    
    # Handles both ipywidgets v7 (dict) and v8 (tuple)
    files = upload_widget.value
    if isinstance(files, tuple):
        for file_item in files:
            name = file_item["name"] if isinstance(file_item, dict) else file_item.name
            content = file_item["content"] if isinstance(file_item, dict) else file_item.content
            filepath = Path(temp_dir) / name
            filepath.write_bytes(bytes(content))
            saved_paths.append(filepath)
    elif isinstance(files, dict):
        for name, data in files.items():
            filepath = Path(temp_dir) / name
            filepath.write_bytes(bytes(data["content"]))
            saved_paths.append(filepath)
            
    return saved_paths


def on_run_clicked(button):
    with output:
        output.clear_output(wait=True)

        if not dxf_chooser.value:
            print("Please select at least one DXF file.")
            return

        # Use a temporary directory to store uploaded file bytes as real files on disk
        with tempfile.TemporaryDirectory() as temp_dir:
            dxf_paths = extract_uploaded_files(dxf_chooser, temp_dir)
            assembly_paths = extract_uploaded_files(assembly_chooser, temp_dir)
            electrode_paths = extract_uploaded_files(electrode_chooser, temp_dir)
            flex_pcb_paths = extract_uploaded_files(flex_pcb_chooser, temp_dir)
            rigid_pcb_paths = extract_uploaded_files(rigid_pcb_chooser, temp_dir)
            headstage_paths = extract_uploaded_files(headstage_chooser, temp_dir)

            try:
                print("1. Loading electrode layout...")
                dxf_file_path = dxf_paths[0]  # Uses the first uploaded DXF file
                xy = utils.load_and_plot_electrodes(
                    dxf_file_path,
                    layer_name=layer_name,
                    num_contours=num_contours,
                )

                print("2. Building Assembly Router & tracing paths...")
                router = AssemblyRouter()

                component_files = [
                    ("electrode", electrode_paths),
                    ("flex_pcb", flex_pcb_paths),
                    ("rigid_pcb", rigid_pcb_paths),
                    ("headstage", headstage_paths),
                ]

                instance_counts = defaultdict(int)
                component_categories = {}

                for category, paths in component_files:
                    for path in paths:
                        definition = ComponentDefinition.from_json(path)
                        router.register_model(definition)

                        instance_id = f"{category}_{instance_counts[category]}"
                        router.add_instance(instance_id, definition.model_name)

                        component_categories[instance_id] = category
                        instance_counts[category] += 1


                for path in assembly_paths:
                    print(f"   Loading Assembly CSV: {path.name}")
                    router.load_assembly_csv(path)

                starts = []

                for instance_id, category in component_categories.items():
                    if category == "electrode":
                        model_name = router.instances[instance_id]
                        definition = router.models[model_name]
                        starts.extend(
                            f"{instance_id}.{channel}"
                            for channel in definition.mapping
                        )

                report = router.report(starts)
                complete_report = [item for item in report if item["status"] == "complete"]

                if not complete_report:
                    print("No complete signal paths discovered by Assembly Router.")
                    return

                print("3. Generating channel map matrix...")
                num_electrodes = len(xy)
                channel_map = np.full((3, num_electrodes), np.nan)

                for item in complete_report:
                    source_pin = item["source"].rsplit(".", 1)[1]
                    destination_pin = item["destination"].rsplit(".", 1)[1]

                    electrode_channel = int(source_pin)
                    headstage_channel = int(destination_pin)
                    electrode_index = electrode_channel - 1

                    if 0 <= electrode_index < num_electrodes:
                        channel_map[0, electrode_index] = xy[electrode_index, 0]
                        channel_map[1, electrode_index] = xy[electrode_index, 1]
                        channel_map[2, electrode_index] = headstage_channel

                state["channel_map"] = channel_map
                state["dxf_file"] = str(dxf_file_path)

                print("\nSuccess!")
                print(f"Shape: {channel_map.shape}")
                print(f"Sample column 0: {channel_map[:, 0]}")

            except Exception as error:
                print(f"Error generating channel map: {error}")

In [6]:
run_button.on_click(on_run_clicked)
display(run_button, output)

Button(button_style='success', description='Generate Channel Map', style=ButtonStyle())

Output()

In [17]:
export_button = widgets.Button(
    description="Export to CSV",
    button_style="info",
)
export_output = widgets.Output()


def on_export_clicked(button):
    with export_output:
        export_output.clear_output(wait=True)

        if state["channel_map"] is None:
            print("Generate a channel map first")
            return

        try:
            export_file = utils.export_channel_map_to_csv(
                state["channel_map"],
                state["dxf_file"],
            )
            print(f"Exported to: {export_file}")
        except Exception as error:
            print(f"Export failed: {error}")


export_button.on_click(on_export_clicked)
display(export_button, export_output)

Button(button_style='info', description='Export to CSV', style=ButtonStyle())

Output()